# 03 — Train Faster R-CNN ResNet-50

Pipeline torchvision Faster R-CNN ResNet-50 trên VNTS, 52 lớp, không crop từng biển. Checkpoint chọn theo mAP50-95 trên val. Online preprocess chỉ chạy khi `ONLINE = True`.


## 0) Bootstrap (Kaggle / Colab)

Clone hoặc pull nhánh, rồi cài dependency. Trên Kaggle bật Internet: On.


In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/tuan8p/VN-Traffic-Sign-Detection.git"
REPO_DIR = "VN-Traffic-Sign-Detection"
REPO_BRANCH = "main"  # Team YOLO đổi thành yolo. Team Faster R-CNN đổi thành faster-rcnn khi nhánh đã có trên origin.

base = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
repo_path = base / REPO_DIR

def sh(cmd: str, check: bool = True) -> int:
    print(">>", cmd)
    return subprocess.run(cmd, shell=True, check=check).returncode

if (Path.cwd() / ".git").exists() and (Path.cwd() / "configs" / "shared.yaml").exists():
    repo_path = Path.cwd()
    print("Using the repo this notebook sits in:", repo_path)
elif not (repo_path / ".git").exists():
    sh(f'git clone --branch {REPO_BRANCH} {REPO_URL} "{repo_path}"')

git = f'git -C "{repo_path}"'
sh(f"{git} fetch origin {REPO_BRANCH}")
sh(f"{git} checkout -B {REPO_BRANCH} origin/{REPO_BRANCH}")

os.chdir(repo_path)
if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))
print("cwd:", Path.cwd())
sh(f"{git} log --oneline -1")

sh('pip install -q -e ".[detect]"')
sh("pip install -q wandb")

import torch
import vn_tsd
from vn_tsd.config.resolve import resolve_config
print(
    f"Bootstrap OK. PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()} | GPUs: {torch.cuda.device_count()}"
)


## 1) Đường dẫn data/processed

Cache offline dùng chung: ảnh letterbox và nhãn YOLO, không crop từng biển. Trên Kaggle gắn dataset bằng Add Input rồi điền vào `CANDIDATES`.


In [ ]:
from pathlib import Path

# After Add Input on Kaggle, paste the dataset path here.
CANDIDATES = [
    # "/kaggle/input/datasets/<user>/vnts-detection-processed",
    "data/processed",
]

def _ready(path: Path) -> bool:
    return (path / "data.yaml").exists() or (path / "images" / "train").exists()

PROCESSED_ROOT = next((c for c in CANDIDATES if c and _ready(Path(c))), None)
if PROCESSED_ROOT is None:
    print("[!] data/processed chưa có data.yaml hoặc images/train.")
    print("    Chạy preprocessing offline trước, hoặc gắn dataset Kaggle vào CANDIDATES.")
    PROCESSED_ROOT = "data/processed"
else:
    print("[OK] Processed root:", PROCESSED_ROOT)


## 2) Cấu hình

Sửa biến ở đây. Batch là tổng của cả hai GPU, không phải mỗi GPU. Không lật ngang.


In [ ]:
# Batch train và eval batch là tổng hai GPU, và phải giống nhau ở cả hai pipeline.
# Chọn số sát VRAM của 2 T4. Hai pipeline phải dùng cùng tổng.
# Đổi batch, eval batch hoặc early_stopping_patience thì báo team kia, rồi sửa cả notebook còn lại và configs/shared.yaml.
# Các biến ở đây chỉ là mức cơ bản. Khi build pipeline, team có thể thêm biến khác cho dễ tune.
PIPELINE = "faster_rcnn"
EPOCHS = 20
LR = 0.005
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0005
BATCH_SIZE = 8
EARLY_STOPPING_PATIENCE = 10
EVAL_BATCH_SIZE = 16
PRETRAINED = True
ONLINE = False          # True chỉ khi đang tune online preprocess
REQUIRE_WANDB = True
USE_AMP = True
PRIMARY_METRIC = "mAP50-95"
SEED = 42
# Tên hiện trên W&B: {pipeline}_s{seed}_ep{epochs}_bs{batch}[_online]
suffix = "_online" if ONLINE else ""
RUN_NAME = f"{PIPELINE}_s{SEED}_ep{EPOCHS}_bs{BATCH_SIZE}{suffix}"


## 3) Resolve config

Ghép biến ở trên với `configs/shared.yaml` và file pipeline. Ghi file overrides để lệnh torchrun bên dưới dùng đúng cấu hình này.


In [ ]:
from pathlib import Path
OVERRIDES = {
    "seed": SEED,
    "model": {"pretrained": PRETRAINED},
    "train": {
        "epochs": EPOCHS,
        "lr": LR,
        "momentum": MOMENTUM,
        "weight_decay": WEIGHT_DECAY,
        "batch_size": BATCH_SIZE,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "amp": USE_AMP,
        "require_wandb": REQUIRE_WANDB,
        "run_name": RUN_NAME,
        "primary_metric": PRIMARY_METRIC,
    },
    "eval": {"batch_size": EVAL_BATCH_SIZE, "auto_eval_test": True, "split": "test"},
    "online_preprocess": {"enabled": ONLINE, "hflip": False},
    "data": {"root": PROCESSED_ROOT},
}


from vn_tsd.config.resolve import resolve_config
from vn_tsd.utils.io import save_yaml

runtime_yaml = "configs/runtime/kaggle.yaml" if Path("/kaggle/working").exists() else "configs/runtime/local.yaml"
cfg = resolve_config(
    pipeline_yaml=f"configs/pipelines/{PIPELINE}.yaml",
    shared_yaml="configs/shared.yaml",
    runtime_yaml=runtime_yaml,
    overrides=OVERRIDES,
)

OVERRIDES_PATH = Path("outputs") / "_notebook" / f"overrides_{PIPELINE}.yaml"
save_yaml(OVERRIDES, OVERRIDES_PATH)

train = cfg.get("train", {})
print("=== Resolved configuration ===")
print("Run name        :", RUN_NAME)
print("Pipeline        :", cfg.get("pipeline"))
print("Epochs          :", train.get("epochs"))
print("Primary metric  :", train.get("primary_metric"))
print("Batch total     :", train.get("batch_size"))
print("Eval batch total:", cfg.get("eval", {}).get("batch_size"))
print("AMP             :", train.get("amp"))
print("W&B required    :", train.get("require_wandb"))
print("Online          :", cfg.get("online_preprocess", {}).get("enabled"))
print("Data root       :", cfg.get("data", {}).get("root"))
print("Overrides file  :", OVERRIDES_PATH)


## 4) Test W&B

Trên Kaggle: Add-ons → Secrets, tên `WANDB_API_KEY`. Host là forge.coreweave.com, team HCMUT_IPCV, project BTL. Tên run trên W&B là `RUN_NAME`. Cell này login rồi ghi một điểm `wandb_smoke`. Fail nghĩa là key hoặc mạng chưa sẵn sàng.


In [ ]:
import os

base_url = (cfg.get("project") or {}).get("wandb_base_url") or "https://forge.coreweave.com"
entity = (cfg.get("project") or {}).get("wandb_entity") or "HCMUT_IPCV"
project = (cfg.get("project") or {}).get("wandb_project") or "BTL"
os.environ["WANDB_BASE_URL"] = base_url

if REQUIRE_WANDB:
    try:
        from kaggle_secrets import UserSecretsClient
        wandb_key = UserSecretsClient().get_secret("WANDB_API_KEY")
        os.environ["WANDB_API_KEY"] = str(wandb_key).strip()
        print("W&B key loaded from Kaggle secrets.")
    except Exception:
        print("No Kaggle secret. Using WANDB_API_KEY from the environment, or an interactive login.")

    import wandb
    if not os.environ.get("WANDB_API_KEY"):
        wandb.login(host=base_url)
    run = wandb.init(
        project=project,
        entity=entity,
        name=RUN_NAME,
        job_type="connectivity-test",
        reinit=True,
    )
    wandb.log({"wandb_smoke": 1})
    url = getattr(run, "url", None)
    run.finish()
    print("W&B test OK:", url)
else:
    os.environ["WANDB_MODE"] = "offline"
    import wandb
    run = wandb.init(project=project, entity=entity, name=RUN_NAME, mode="offline", reinit=True)
    wandb.log({"wandb_smoke": 1})
    run.finish()
    print("W&B test OK (offline).")


## 5) Train

Mặc định là Kaggle 2× T4: `torchrun --nproc_per_node=2` và `configs/runtime/kaggle.yaml`. Local chỉ thử xem có chạy được không, dùng `python -m tools.run_train` và `configs/runtime/local.yaml`.

`fit()` hiện vẫn là khung. Team pipeline điền model thì checkpoint mới ra.


In [ ]:
import subprocess
from pathlib import Path

on_kaggle = Path("/kaggle/working").exists()
runtime = "configs/runtime/kaggle.yaml" if on_kaggle else "configs/runtime/local.yaml"
cmd = [
    "--pipeline", PIPELINE,
    "--runtime", runtime,
    "--run-name", RUN_NAME,
    "--overrides", str(OVERRIDES_PATH),
    "--data-root", PROCESSED_ROOT,
    "--online" if ONLINE else "--no-online",
]
if on_kaggle:
    cmd = ["torchrun", "--nproc_per_node=2", "-m", "tools.run_train", *cmd]
else:
    cmd = ["python", "-m", "tools.run_train", *cmd]
print(">>", " ".join(cmd))
subprocess.run(cmd, check=True)
